In [ ]:
import pandas as pd
from pyspark.sql.functions import to_date, col
from datetime import datetime, timedelta
import random
import math

# Simplified Static Data
equipment_data = [
    {"equipment_id": "EQ001", "name": "CNC Machine Alpha", "location": "Floor A", "status": "Active"},
    {"equipment_id": "EQ002", "name": "Robotic Arm Beta", "location": "Floor A", "status": "Active"},
    {"equipment_id": "EQ003", "name": "Conveyor Gamma", "location": "Floor B", "status": "Maintenance"}
]

production_line_data = [
    {"line_id": "LINE001", "name": "Assembly Line Alpha", "department": "Manufacturing"},
    {"line_id": "LINE002", "name": "Assembly Line Beta", "department": "Manufacturing"},
    {"line_id": "LINE003", "name": "Quality Control Line", "department": "Quality"}
]

equipment_line_relationships = [
    {"equipment_id": "EQ001", "line_id": "LINE001"},
    {"equipment_id": "EQ002", "line_id": "LINE001"},
    {"equipment_id": "EQ003", "line_id": "LINE002"}
]

# Generate Simplified Time Series Data
def generate_timeseries_data():
    equipment_data = []
    production_data = []
    base_time = datetime(2024, 1, 1, 0, 0, 0)
    
    for day in range(3):  # Just 7 days
        for hour in range(0, 24, 4):  # Every 4 hours
            timestamp = base_time + timedelta(days=day, hours=hour)
            
            # Equipment metrics - just temperature and efficiency
            for eq_id in ["EQ001", "EQ002", "EQ003"]:
                equipment_data.append({
                    "timestamp": timestamp,
                    "equipment_id": eq_id,
                    "temperature": float(f"{60 + random.uniform(-10, 15):.1f}"),
                    "efficiency": float(f"{85 + random.uniform(-10, 10):.1f}")
                })
            
            # Production line metrics - just units and quality
            for line_id in ["LINE001", "LINE002", "LINE003"]:
                production_data.append({
                    "timestamp": timestamp,
                    "line_id": line_id,
                    "units_produced": random.randint(30, 60),
                    "quality_score": float(f"{95 + random.uniform(-3, 3):.1f}")
                })
    
    return equipment_data, production_data

print("Generating simplified data...")
equipment_ts_data, production_ts_data = generate_timeseries_data()

# Create DataFrames
equipment_df = spark.createDataFrame(equipment_data)
production_line_df = spark.createDataFrame(production_line_data)
relationships_df = spark.createDataFrame(equipment_line_relationships)
equipment_ts_df = spark.createDataFrame(equipment_ts_data).withColumn("date", to_date(col("timestamp")))
production_ts_df = spark.createDataFrame(production_ts_data).withColumn("date", to_date(col("timestamp")))

# Write to tables
equipment_df.write.mode("overwrite").saveAsTable("equipment")
production_line_df.write.mode("overwrite").saveAsTable("production_lines")
relationships_df.write.mode("overwrite").saveAsTable("equipment_assignments")
equipment_ts_df.write.mode("overwrite").partitionBy("date").saveAsTable("equipment_metrics")
production_ts_df.write.mode("overwrite").partitionBy("date").saveAsTable("production_metrics")

print("Tables created:")
print(f"- equipment: {equipment_df.count()} rows")
print(f"- production_lines: {production_line_df.count()} rows")
print(f"- equipment_assignments: {relationships_df.count()} rows")
print(f"- equipment_metrics: {equipment_ts_df.count()} rows")
print(f"- production_metrics: {production_ts_df.count()} rows")